# Notebook 3 : Fine-tuning du LLM avec Unsloth et LoRA

Ce notebook fine-tune un LLM pré-entraîné sur le dataset SFT multi-tâches.

L’entraînement est réalisé avec Unsloth et LoRA/QLoRA 

Les exemples Task A, Task B et Task C sont mélangés dans le même dataset d’entraînement.

Le modèle apprend donc à adapter sa réponse selon l’instruction donnée dans le prompt.

**Sorties**

Ce notebook sauvegarde l’adaptateur LoRA fine-tuné et les artefacts d’entraînement :

models/enthymemes_unsloth_lora/

Cet adaptateur est ensuite chargé dans le notebook suivant pour l’inférence et l’évaluation.


### Chargement

In [1]:
!pip install -U unsloth
!pip install -U datasets trl peft accelerate bitsandbytes

  Using cached unsloth-2026.6.2-py3-none-any.whl.metadata (58 kB)
  Using cached unsloth_zoo-2026.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached torch-2.10.0-cp311-cp311-win_amd64.whl.metadata (31 kB)
  Using cached tyro-1.0.13-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-win_amd64.whl.metadata (1.0 kB)
  Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached triton_windows-3.7.0.post26-cp311-cp311-win_amd64.whl.metadata (1.8 kB)
  Using cached sentencepiece-0.2.1-cp311-cp311-win_amd64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached huggingface_hub-1.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-win_amd64.whl.metadata (1.8 kB)
  Using cached diffusers-0.38.0-py3-none-any.whl.metadata (20 kB)
  Using cached


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached trl-1.5.1-py3-none-any.whl.metadata (11 kB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
Using cached trl-1.5.1-py3-none-any.whl (761 kB)

  Attempting uninstall: datasets

    Found existing installation: datasets 4.3.0

    Uninstalling datasets-4.3.0:

      Successfully uninstalled datasets-4.3.0

   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datasets]
  Attempting uninstall: trl
   ---------------------------------------- 0/2 [datasets]
    Found existing installation: trl 0.24.0
   ---------------------------------------- 0/2 [datasets]
    Uninstalling trl-0.24.0:
   ---------------------------------------- 0/2 [datasets]
      Successfully uninstalled trl-0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.6.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth 2026.6.2 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.5.1 which is incompatible.
unsloth-zoo 2026.6.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth-zoo 2026.6.2 requires trl!=0.19.0,<=0.24.0,>=0.18.2; sys_platform != "darwin" or platform_machine != "arm64", but you have trl 1.5.1 which is incompatible.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

import pandas as pd
from datasets import Dataset

from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

from trl import SFTTrainer, SFTConfig

2.10.0+cu130
13.0
True
NVIDIA GeForce RTX 5070
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0611 11:13:09.818000 11380 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected. Unsloth fine-tuning requires CUDA GPU.")

CUDA available: True
GPU: NVIDIA GeForce RTX 5070
CUDA version: 13.0


In [3]:
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

SFT_DIR = Path("data/sft")
OUTPUT_DIR = Path("models/enthymemes_unsloth_lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SFT_DIR / "train_sft.jsonl"
VALIDATION_PATH = SFT_DIR / "validation_sft.jsonl"
TEST_PATH = SFT_DIR / "test_sft.jsonl"

In [4]:
# Рекомендованный стартовый вариант.
# Qwen обычно проще использовать, чем Llama, потому что Llama может требовать Hugging Face access token.
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
# MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
# MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# LoRA settings
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0



### Chargement SFT dataset

In [5]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return records

In [6]:
train_sft = read_jsonl(TRAIN_PATH)
validation_sft = read_jsonl(VALIDATION_PATH)
test_sft = read_jsonl(TEST_PATH)

print("Train SFT:", len(train_sft))
print("Validation SFT:", len(validation_sft))
print("Test SFT:", len(test_sft))

Train SFT: 3932
Validation SFT: 502
Test SFT: 498


In [7]:
train_sft[0].keys()

dict_keys(['messages', 'text_preview', 'metadata'])

### Check structure des exemples

In [8]:
example = train_sft[0]

print("Metadata:")
print(json.dumps(example["metadata"], ensure_ascii=False, indent=2))

print("\nMessages:")
for message in example["messages"]:
    print("=" * 80)
    print("ROLE:", message["role"])
    print(message["content"][:1000])

Metadata:
{
  "tweet_id": "663",
  "split": "train",
  "task_type": "task_a_classification_uncertainty",
  "majority_label": "premise",
  "uncertainty_level": "low",
  "usable_for_hard_classification_eval": true
}

Messages:
ROLE: system
You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.

ROLE: user
Task A: Analyze the enthymeme and classify the missing implicit argumentation.

You must decide whether the tweet contains missing implicit argumentation.
If yes, classify the missing argument as:
- premise
- claim

If there is no missing implicit argumentation, use:
- none

If the historical annota

### Stats of train exemples

In [9]:
def sft_metadata_to_df(examples: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []

    for ex in examples:
        row = ex["metadata"].copy()
        rows.append(row)

    return pd.DataFrame(rows)

In [10]:
train_meta_df = sft_metadata_to_df(train_sft)
validation_meta_df = sft_metadata_to_df(validation_sft)

display(train_meta_df.head())

print("Task distribution:")
display(train_meta_df["task_type"].value_counts())

print("Majority label distribution:")
display(train_meta_df["majority_label"].value_counts())

print("Uncertainty distribution:")
display(train_meta_df["uncertainty_level"].value_counts())

,tweet_id,split,task_type,majority_label,uncertainty_level,usable_for_hard_classification_eval,usable_for_full_pipeline_eval,annotation_label,annotator_id
0,663,train,task_a_classification_uncertainty,premise,low,True,NaN,NaN,NaN
1,95,train,task_c_full_pipeline,none,low,NaN,True,NaN,NaN
2,1793,train,task_b_generation,none,medium,NaN,NaN,premise,4.0
3,1331,train,task_a_classification_uncertainty,none,low,True,NaN,NaN,NaN
4,65,train,task_a_classification_uncertainty,none,low,True,NaN,NaN,NaN


Task distribution:


task_type
task_b_generation                    1825
task_a_classification_uncertainty    1066
task_c_full_pipeline                 1041
Name: count, dtype: int64

Majority label distribution:


majority_label
premise      1901
none         1682
claim         242
ambiguous     107
Name: count, dtype: int64

Uncertainty distribution:


uncertainty_level
low       3210
medium     615
high       107
Name: count, dtype: int64

In [11]:
# TEST RUN

DEBUG_RUN = False

if DEBUG_RUN:
    train_sft = train_sft[:100]
    validation_sft = validation_sft[:30]

print("Train examples used:", len(train_sft))
print("Validation examples used:", len(validation_sft))

Train examples used: 3932
Validation examples used: 502


### Chargement de modèle

Chargement de base model avec Unsloth

In [12]:
#charge the model

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

==((====))==  Unsloth 2026.6.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5070. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [13]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer pad token:", tokenizer.pad_token)
print("Tokenizer eos token:", tokenizer.eos_token)

Tokenizer pad token: <|PAD_TOKEN|>
Tokenizer eos token: <|im_end|>


### LoRA

In [14]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.6.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Formatage des données avec le chat template

Dans le notebook précédent, les données ont été sauvegardées sous forme de messages :

```text
system / user / assistant

Avant l’entraînement, ces messages doivent être convertis en une seule séquence de texte.
Pour cela, on applique le chat template du tokenizer du modèle choisi.

In [15]:
def apply_chat_template_to_example(example: Dict[str, Any]) -> Dict[str, Any]:
    messages = example["messages"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "text": text,
        "metadata": example["metadata"],
    }

In [16]:
train_text_records = [
    apply_chat_template_to_example(ex)
    for ex in train_sft
]

validation_text_records = [
    apply_chat_template_to_example(ex)
    for ex in validation_sft
]

test_text_records = [
    apply_chat_template_to_example(ex)
    for ex in test_sft
]

print(train_text_records[0]["text"][:1500])

<|im_start|>system
You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.
<|im_end|>
<|im_start|>user
Task A: Analyze the enthymeme and classify the missing implicit argumentation.

You must decide whether the tweet contains missing implicit argumentation.
If yes, classify the missing argument as:
- premise
- claim

If there is no missing implicit argumentation, use:
- none

If the historical annotations are strongly divided, mark the case as ambiguous.

Tweet:
You can see here eye witness testimony of the original trial volunteers - this one a doctor herself - saying how they were wiped off the tr

### Preparer le Dataset

In [17]:
train_dataset = Dataset.from_list(train_text_records)
validation_dataset = Dataset.from_list(validation_text_records)
test_dataset = Dataset.from_list(test_text_records)

train_dataset

Dataset({
    features: ['text', 'metadata'],
    num_rows: 3932
})

In [18]:
print(train_dataset[0]["text"][:1500])

<|im_start|>system
You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.
<|im_end|>
<|im_start|>user
Task A: Analyze the enthymeme and classify the missing implicit argumentation.

You must decide whether the tweet contains missing implicit argumentation.
If yes, classify the missing argument as:
- premise
- claim

If there is no missing implicit argumentation, use:
- none

If the historical annotations are strongly divided, mark the case as ambiguous.

Tweet:
You can see here eye witness testimony of the original trial volunteers - this one a doctor herself - saying how they were wiped off the tr

### Check longeur de sequence

In [19]:
def count_tokens(text: str) -> int:
    return len(
        tokenizer(
            text,
            truncation=False,
            add_special_tokens=False,
        )["input_ids"]
    )

In [20]:
sample_size = min(200, len(train_text_records))

lengths = [
    count_tokens(train_text_records[i]["text"])
    for i in range(sample_size)
]

lengths_df = pd.DataFrame({"n_tokens": lengths})

display(lengths_df.describe())

print("Max tokens in sample:", max(lengths))
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)

,n_tokens
count,200.000000
mean,323.155000
std,63.709856
min,224.000000
25%,263.750000
50%,338.500000
75%,376.250000
max,518.000000


Max tokens in sample: 518
MAX_SEQ_LENGTH: 2048


### Training configuration 

In [ ]:
# Nombre d'exemples traités en même temps par le GPU
PER_DEVICE_BATCH_SIZE = 2

# Nombre d'étapes pendant lesquelles les gradients sont accumulés
# avant de mettre à jour les paramètres du modèle
GRADIENT_ACCUMULATION_STEPS = 4

# Taux d'apprentissage utilisé pour mettre à jour les adaptateurs LoRA
LEARNING_RATE = 2e-4

# Nombre de passages complets sur le dataset d'entraînement
NUM_TRAIN_EPOCHS = 3

# Fréquence d'affichage des informations d'entraînement
LOGGING_STEPS = 10

# Fréquence d'évaluation sur le validation set
EVAL_STEPS = 50

# Fréquence de sauvegarde des checkpoints du modèle
SAVE_STEPS = 100

In [22]:
fp16 = not is_bfloat16_supported()
bf16 = is_bfloat16_supported()

print("fp16:", fp16)
print("bf16:", bf16)

fp16: False
bf16: True


### SFT configuration

In [ ]:
training_args = SFTConfig(
    # Dossier où seront sauvegardés les checkpoints et les résultats d'entraînement
    output_dir=str(OUTPUT_DIR),

    # Nom de la colonne contenant le texte final utilisé pour l'entraînement
    dataset_text_field="text",

    # Longueur maximale des séquences en tokens
    max_seq_length=MAX_SEQ_LENGTH,

    # Désactive le packing : chaque exemple reste séparé
    packing=False,

    # Nombre d'exemples traités en même temps par GPU pendant l'entraînement
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,

    # Nombre d'exemples traités en même temps par GPU pendant l'évaluation
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,

    # Accumule les gradients sur plusieurs étapes avant une mise à jour du modèle
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Nombre de passages complets sur le dataset d'entraînement
    num_train_epochs=NUM_TRAIN_EPOCHS,

    # Taux d'apprentissage utilisé pour mettre à jour les paramètres LoRA
    learning_rate=LEARNING_RATE,

    # Phase de démarrage progressif du learning rate au début de l'entraînement
    warmup_ratio=0.03,

    # Régularisation pour limiter le surapprentissage
    weight_decay=0.01,

    # Le learning rate diminue progressivement de manière linéaire
    lr_scheduler_type="linear",

    # Optimiseur AdamW en 8-bit pour réduire l'utilisation de la mémoire
    optim="adamw_8bit",

    # Fréquence d'affichage des logs d'entraînement
    logging_steps=LOGGING_STEPS,

    # Évaluation régulière pendant l'entraînement
    eval_strategy="steps",

    # Fréquence d'évaluation sur le validation set
    eval_steps=EVAL_STEPS,

    # Sauvegarde régulière pendant l'entraînement
    save_strategy="steps",

    # Fréquence de sauvegarde des checkpoints
    save_steps=SAVE_STEPS,

    # Nombre maximal de checkpoints conservés
    save_total_limit=2,

    # Utilise fp16 si bf16 n'est pas disponible
    fp16=fp16,

    # Utilise bf16 si le GPU le supporte
    bf16=bf16,

    # Graine aléatoire pour rendre l'entraînement plus reproductible
    seed=SEED,

    # Désactive l'envoi des logs vers des services externes comme WandB
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [24]:
try:
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        args=training_args,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        args=training_args,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
    )

Unsloth: Tokenizing ["text"]:   0%|          | 0/3932 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/502 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


### Training 

In [25]:
# TRAIN

trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,932 | Num Epochs = 3 | Total steps = 1,476
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.585525,0.581868
100,0.494142,0.540375
150,0.529420,0.531136
200,0.455572,0.528139
250,0.485804,0.532433
300,0.406968,0.544747
350,0.376308,0.560020
400,0.371032,0.564762
450,0.323003,0.580565
500,0.262671,0.627667


Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\checkpoint-100\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\checkpoint-200\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\checkpoint-300\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\checkpoint-400\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\checkpoint-500\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\checkpoint-600\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\checkpoint-700\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\checkpoint-800\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder m

In [29]:
#EVAL

from transformers.utils.notebook import NotebookProgressCallback

trainer.remove_callback(NotebookProgressCallback)

eval_results = trainer.evaluate()
eval_results

#eval_results = trainer.evaluate()
#eval_results


{'eval_loss': 0.8696860671043396,
 'eval_runtime': 30.9329,
 'eval_samples_per_second': 16.229,
 'eval_steps_per_second': 8.114,
 'epoch': 3.0}

### Sauvegarder LoRA

In [30]:
#SAVE 

LORA_OUTPUT_DIR = OUTPUT_DIR / "lora_adapter"
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

print("Saved LoRA adapter to:", LORA_OUTPUT_DIR)

Unsloth: Restored added_tokens_decoder metadata in models\enthymemes_unsloth_lora\lora_adapter\tokenizer_config.json.


Saved LoRA adapter to: models\enthymemes_unsloth_lora\lora_adapter


### Sauvegarder les stats 

In [31]:
stats_path = OUTPUT_DIR / "trainer_stats.json"

try:
    stats_dict = trainer_stats.metrics
except Exception:
    stats_dict = {}

with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats_dict, f, ensure_ascii=False, indent=2)

print("Saved stats to:", stats_path)

Saved stats to: models\enthymemes_unsloth_lora\trainer_stats.json


### Inference test

In [32]:
#Inference test 

FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

### Prompt task A

In [33]:
# Prompt task A

SYSTEM_PROMPT = """You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.
"""

def build_task_a_prompt(tweet: str) -> List[Dict[str, str]]:
    user_prompt = f"""Task A: Analyze the enthymeme and classify the missing implicit argumentation.

You must decide whether the tweet contains missing implicit argumentation.
If yes, classify the missing argument as:
- premise
- claim

If there is no missing implicit argumentation, use:
- none

If the case is strongly uncertain, mark it as ambiguous.

Tweet:
{tweet}

Return JSON with the following fields:
- argument_analysis
- implicit_status: yes / no / ambiguous
- implicit_type: none / premise / claim / ambiguous
- vote_distribution
- confidence
- uncertainty_level
"""

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

### Prompt task B

In [34]:
# Prompt task B

def build_task_b_prompt(
    tweet: str,
    implicit_type: str,
    uncertainty_level: str = "unknown",
    vote_distribution: Optional[Dict[str, float]] = None,
) -> List[Dict[str, str]]:

    if vote_distribution is None:
        vote_distribution = {
            "none": 0.0,
            "premise": 0.0,
            "claim": 0.0,
        }

    user_prompt = f"""Task B: Generate the missing implicit argument.

The implicit argument type is given.
Generate only the missing proposition that completes the enthymeme.

Tweet:
{tweet}

Implicit argument type:
{implicit_type}

Model-estimated vote distribution:
none: {vote_distribution.get("none", 0.0)}
premise: {vote_distribution.get("premise", 0.0)}
claim: {vote_distribution.get("claim", 0.0)}

Model-estimated uncertainty level:
{uncertainty_level}

Return JSON with the following fields:
- implicit_type
- implicit_argument
"""

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

In [35]:
#Fonction de généralisation 

def generate_from_messages(
    messages: List[Dict[str, str]],
    max_new_tokens: int = 512,
    temperature: float = 0.1,
    do_sample: bool = False,
) -> str:
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return generated_text.strip()

In [36]:
#json parser 

def try_parse_json(text: str) -> Optional[Dict[str, Any]]:
    """
    Пытается достать JSON из ответа модели.
    """

    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    # fallback: ищем первый JSON object
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]
        try:
            return json.loads(candidate)
        except Exception:
            return None

    return None

### Test sur 1 exemple de validation

In [37]:
#test 

val_example = validation_sft[0]

tweet = None

for message in val_example["messages"]:
    if message["role"] == "user":
        user_content = message["content"]
        break

print(user_content[:1500])

Task B: Generate the missing implicit argument.

The implicit argument type is given.
Generate only the missing proposition that completes the enthymeme.

Tweet:
This POS preaching about choice after mask mandates, (even for children) vaccine mandates and shutting down schools and businesses. Getting medical personnel fired after they were on the front wall? FH.

Implicit argument type:
premise

Historical vote distribution:
none: 0.0
premise: 0.8
claim: 0.2

Historical uncertainty level:
low

Return JSON with the following fields:
- implicit_type
- implicit_argument



Juste un test rapide donc on prend un exemple à la main

In [38]:
tweet = "#AsiaBibi's acquittal has been upheld by Pakistan's supreme court. She is still in fear for her life however. Remember that the UK refused this Christian woman asylum because of fears it would cause \"unrest\". An indelible disgrace on our government"

task_a_messages = build_task_a_prompt(tweet)
task_a_raw = generate_from_messages(task_a_messages)

print(task_a_raw)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\jeann\Documents\Projets_Python\Deep_Learning\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\jeann\Documents\Projets_Python\Deep_Learning\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
 

{
  "argument_analysis": "The tweet contains an explicit evaluative or argumentative statement. The missing proposition functions as a supporting premise: A country that refuses asylum to a persecuted person due to fears of domestic unrest is acting with racist motives",
  "implicit_status": "yes",
  "implicit_type": "premise",
  "vote_distribution": {
    "none": 0.2,
    "premise": 0.8,
    "claim": 0.0
  },
  "confidence": 0.8,
  "uncertainty_level": "low"
}


In [39]:
task_a_json = try_parse_json(task_a_raw)
task_a_json

{'argument_analysis': 'The tweet contains an explicit evaluative or argumentative statement. The missing proposition functions as a supporting premise: A country that refuses asylum to a persecuted person due to fears of domestic unrest is acting with racist motives',
 'implicit_status': 'yes',
 'implicit_type': 'premise',
 'vote_distribution': {'none': 0.2, 'premise': 0.8, 'claim': 0.0},
 'confidence': 0.8,
 'uncertainty_level': 'low'}

### Two step inference pipeline

In [ ]:
#two step inference 

def predict_enthymeme(tweet: str) -> Dict[str, Any]:
    """
    Final inference pipeline:
    1. Task A: classification + uncertainty
    2. Task B: generation, if implicit_type = premise or claim
    """

    # Step 1: classification
    task_a_messages = build_task_a_prompt(tweet)
    task_a_raw = generate_from_messages(
        task_a_messages,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
    )

    task_a_json = try_parse_json(task_a_raw)

    if task_a_json is None:
        return {
            "error": "Could not parse Task A JSON",
            "task_a_raw": task_a_raw,
        }

    implicit_type = task_a_json.get("implicit_type", "none")
    uncertainty_level = task_a_json.get("uncertainty_level", "unknown")
    vote_distribution = task_a_json.get(
        "vote_distribution",
        {
            "none": 0.0,
            "premise": 0.0,
            "claim": 0.0,
        },
    )

    # Step 2: no generation for none or ambiguous
    if implicit_type in ["none", "ambiguous"]:
        return {
            **task_a_json,
            "implicit_argument": "",
            "task_a_raw": task_a_raw,
            "task_b_raw": "",
        }

    # Step 3: generation
    task_b_messages = build_task_b_prompt(
        tweet=tweet,
        implicit_type=implicit_type,
        uncertainty_level=uncertainty_level,
        vote_distribution=vote_distribution,
    )

    task_b_raw = generate_from_messages(
        task_b_messages,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False,
    )

    task_b_json = try_parse_json(task_b_raw)

    if task_b_json is None:
        implicit_argument = ""
    else:
        implicit_argument = task_b_json.get("implicit_argument", "")

    return {
        **task_a_json,
        "implicit_argument": implicit_argument,
        "task_a_raw": task_a_raw,
        "task_b_raw": task_b_raw,
    }

In [41]:
#check 

result = predict_enthymeme(tweet)

print(json.dumps(result, ensure_ascii=False, indent=2))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "argument_analysis": "The tweet contains an explicit evaluative or argumentative statement. The missing proposition functions as a supporting premise: A country that refuses asylum to a persecuted person due to fears of domestic unrest is acting with racist motives",
  "implicit_status": "yes",
  "implicit_type": "premise",
  "vote_distribution": {
    "none": 0.2,
    "premise": 0.8,
    "claim": 0.0
  },
  "confidence": 0.8,
  "uncertainty_level": "low",
  "implicit_argument": "A government that refuses asylum to a persecuted person due to fears of domestic unrest is acting disgracefully",
  "task_a_raw": "{\n  \"argument_analysis\": \"The tweet contains an explicit evaluative or argumentative statement. The missing proposition functions as a supporting premise: A country that refuses asylum to a persecuted person due to fears of domestic unrest is acting with racist motives\",\n  \"implicit_status\": \"yes\",\n  \"implicit_type\": \"premise\",\n  \"vote_distribution\": {\n    \"

### Save config

In [42]:
config = {
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "seed": SEED,
}

config_path = OUTPUT_DIR / "training_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved config to:", config_path)

Saved config to: models\enthymemes_unsloth_lora\training_config.json
